# AR-HEFT: Reproducible Reliability-Aware Workflow Scheduling Research Framework

**Purpose:** a self-contained, testable simulation framework for evaluating adaptive reliability-aware scheduling under heterogeneous VM outages and correlated shocks.

**Important scientific boundary:** passing tests means selected implementation invariants hold; it does not establish novelty, realism, or publication readiness. The simulator is a controlled synthetic model. Before a paper submission, validate the assumptions against trace-grounded or benchmark-based workloads, compare with validated published baselines, and report limitations.

### Experiment protocol
1. Run configuration and model cells in order.
2. Run the simulator validation cell. Do not proceed if any assertion fails.
3. Run the smoke test and inspect its output.
4. The main run is configured for 100 matched seeds by default. Set `RUN_MAIN_EXPERIMENT = False` only when you want a quick validation-only pass.
5. Inspect raw runs, summary tables, paired confidence intervals, and multiplicity-corrected tests before writing conclusions.

The simulator uses a deterministic event queue (no SimPy dependency). Every scheduler receives the same workflow and exogenous outage trace for each matched seed/configuration.

## Research status and publication boundary

This notebook is a reproducible **simulation framework**, not a publication-ready evidence package by itself. Synthetic DAGs are useful for debugging and controlled sensitivity tests, but final claims require evaluation on publicly documented scientific workflow benchmarks (for example, WfCommons/Pegasus workflow structures), a transparent parameter table, and comparison against faithful implementations or clearly labeled adapted baselines. Do not describe a literature-only method as implemented.

### Recent related work to read and compare

1. **Yang, Zeng, and Yang (2026), “TFR-GNN: Topology- and Fault-Aware Graph Neural Scheduling for Heterogeneous Distributed Computing Systems,” arXiv:2609.09165.** A learned graph-based scheduler incorporating workflow dependencies, machine topology, reliability tilt, and optional replication. It uses WfCommons/Pegasus workflow structures. It is a strong related-work comparator, but its results must not be copied into this simulator as if reproduced.
2. **Wang et al. (2026), “Reliable cloud workflow scheduling with uncertain task execution time,” Journal of King Saud University Computer and Information Sciences, DOI: 10.1007/s44443-026-00585-x.** Jointly treats deadline constraints, reliability, execution-time uncertainty, and elastic provisioning.
3. **Oikonomou, Kolomvatsos, and Anagnostopoulos (2025), “A proactive and uncertainty driven management mechanism for workflows of processing tasks,” Computing 107, 112, DOI: 10.1007/s00607-025-01468-1.** Integrates uncertainty-aware HEFT with checkpointing and migration for unreliable/preemptible resources.
4. **Mushtaq et al. (2025), “CRFTS: a cluster-centric and reservation-based fault-tolerant scheduling strategy to enhance QoS in cloud computing,” Scientific Reports 15, 32233, DOI: 10.1038/s41598-025-17609-7.** Reservation- and cluster-oriented fault tolerance; its workload assumptions differ from this DAG simulator.
5. **“A robust fault-tolerant framework for VM failure prediction and efficient task scheduling in dynamic cloud environments” (2025), Journal of Network and Computer Applications 244, 104340, DOI: 10.1016/j.jnca.2025.104340.** A failure-prediction and scheduling framework; consult the full paper before claiming exact algorithmic comparability.

### Defensible positioning (subject to literature review)

“AR-HEFT investigates a lightweight, training-free HEFT-style placement policy that updates per-VM hazard estimates from observed outages and combines expected interruption cost with workflow-deadline feasibility. Its contribution, if supported by benchmark evaluation, is an interpretable online heuristic and a measured resilience–makespan trade-off under heterogeneous and correlated outages—not a claim to be the first reliability-aware scheduler or universally superior to learned methods.”

### Baseline status

- `HEFT-ECT`: HEFT-style upward-rank task ordering with earliest-completion placement; it omits inter-task communication costs and is **not canonical HEFT**.
- `STATIC-RA-Oracle`: a diagnostic upper-bound-style baseline that uses the simulator's mode-specific nominal failure rates. It has privileged knowledge of the synthetic failure regime; do not compare it as if it were a deployable scheduler.
- `AR-HEFT`: adaptive hazard estimate with deadline-feasibility preference.
- `AR-HEFT-NoRisk`: deadline-aware ablation with the reliability penalty removed.
- `AR-HEFT-NoDeadline`: adaptive-risk ablation without deadline preference.

A true paper comparison to TFR-GNN, uHEFT, CRFTS, and RWSUT requires either faithful implementation under shared assumptions or a literature-only comparison table. This notebook does not implement those published methods.


In [ ]:
# Cell 1 — Imports and configuration
from __future__ import annotations
import heapq
import math
import time
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

MASTER_SEED = 20261022
VM_SPEEDS = [1.00, 1.20, 0.82, 1.45, 0.95, 1.12, 0.72, 1.30]
BASE_FAILURE_RATE = 0.012       # ordinary outages per unit of healthy uptime
REPAIR_TIME = 3.0               # fixed simulated-time repair duration
RESTART_OVERHEAD = 0.25          # simulated dispatch/reinitialization delay after lost work
PRIOR_RATE = BASE_FAILURE_RATE   # common, non-oracle prior for adaptive hazard estimates
SHOCK_VMS = (0, 1, 2)            # VMs hit by the correlated shock
SHOCK_START_FRACTION = 0.40
SHOCK_DURATION_FRACTION = 0.18
PRIOR_EXPOSURE = 30.0            # Gamma-Poisson prior exposure, time units
RISK_WEIGHT = 8.0
ADAPTIVE_WEIGHT_KAPPA = 0.35
MAX_ADAPTIVE_WEIGHT_MULTIPLIER = 2.5
DEADLINE_SLACK_GUARD_FRACTION = 0.15
DEADLINE_WEIGHT = 2.0
WORKLOAD_SIZES = (12, 24, 48)
WORKFLOW_STRUCTURES = ("chain", "fork_join", "random")
FAILURE_MODES = ("stable", "hotspot", "shock")
SCHEDULERS = ("HEFT-ECT", "STATIC-RA-Oracle", "AR-HEFT", "AR-HEFT-NoRisk", "AR-HEFT-NoDeadline")

# Full experiment defaults: start small, validate, then scale.
RUN_MAIN_EXPERIMENT = True
N_SEEDS_MAIN = 100
DEADLINE_FACTOR_MAIN = 1.05
RUN_DEADLINE_SENSITIVITY = False
N_SEEDS_SENSITIVITY = 5
DEADLINE_FACTORS = (0.90, 1.00, 1.05, 1.15, 1.35)

assert len(VM_SPEEDS) >= 3 and all(v > 0 for v in VM_SPEEDS)
assert BASE_FAILURE_RATE >= 0 and REPAIR_TIME > 0
print("Configuration loaded. Schedulers:", SCHEDULERS)

## Experimental validity checklist

Before interpreting results, the simulator explicitly distinguishes:
- **failure occurrence** (an outage begins),
- **task interruption** (an outage hits a running task),
- **lost work** (progress since the last restart is discarded),
- **recovery delay** (interruption to the task's next start, including restart overhead),
- **workflow deadline violation** (final makespan exceeds the configured deadline).

An interrupted task is restarted from its full original work requirement; it does not silently resume from the interrupted progress. Every scheduler receives the same DAG, deadline, and pre-generated outage trace for a matched seed/configuration. The adaptive scheduler starts from a common prior rate rather than being initialized with the hidden failure-mode-specific rates.


## Workload model
Each task has a positive abstract work requirement and a DAG of dependencies. Communication costs are intentionally omitted; therefore HEFT-ECT is a HEFT-style computation-only baseline, not a claim of canonical HEFT equivalence. Three deterministic DAG families are provided for controlled testing.

In [ ]:
# Cell 2 — Workflow generators and DAG validation
STRUCTURE_CODE = {"chain": 11, "fork_join": 23, "random": 37}

@dataclass
class Task:
    task_id: int
    work: float
    parents: Tuple[int, ...]
    children: Tuple[int, ...] = ()


def make_workflow(structure: str, n_tasks: int, seed: int) -> Dict[int, Task]:
    if structure not in STRUCTURE_CODE:
        raise ValueError(f"Unknown structure: {structure}")
    if n_tasks < 2:
        raise ValueError("n_tasks must be at least 2")
    rng = np.random.default_rng(seed)
    edges = set()
    if structure == "chain":
        edges = {(i, i + 1) for i in range(n_tasks - 1)}
    elif structure == "fork_join":
        # A source, parallel middle layer, and a sink; extra tasks are layered.
        if n_tasks == 2:
            edges = {(0, 1)}
        else:
            middle = list(range(1, n_tasks - 1))
            edges.update((0, j) for j in middle)
            edges.update((j, n_tasks - 1) for j in middle)
    else:
        # Always include a chain backbone to guarantee connectivity; add forward edges.
        edges = {(i, i + 1) for i in range(n_tasks - 1)}
        for i in range(n_tasks):
            for j in range(i + 2, n_tasks):
                if rng.random() < min(0.18, 2.0 / n_tasks):
                    edges.add((i, j))
    parents = {i: [] for i in range(n_tasks)}
    children = {i: [] for i in range(n_tasks)}
    for a, b in sorted(edges):
        children[a].append(b)
        parents[b].append(a)
    works = rng.uniform(4.0, 12.0, size=n_tasks)
    tasks = {
        i: Task(i, float(works[i]), tuple(sorted(parents[i])), tuple(sorted(children[i])))
        for i in range(n_tasks)
    }
    validate_workflow(tasks)
    return tasks


def validate_workflow(tasks: Dict[int, Task]) -> None:
    ids = set(tasks)
    if not ids:
        raise ValueError("Workflow is empty")
    for tid, task in tasks.items():
        if task.task_id != tid or task.work <= 0:
            raise ValueError("Invalid task id or nonpositive work")
        if any(p not in ids or p >= tid for p in task.parents):
            raise ValueError("Workflow must be acyclic with parent IDs preceding children")
        if any(c not in ids or c <= tid for c in task.children):
            raise ValueError("Invalid child edge")
        for p in task.parents:
            if tid not in tasks[p].children:
                raise ValueError("Parent/child edges are inconsistent")
        for c in task.children:
            if tid not in tasks[c].parents:
                raise ValueError("Child/parent edges are inconsistent")


def upward_ranks(tasks: Dict[int, Task], speeds: List[float]) -> Dict[int, float]:
    avg_speed = float(np.mean(speeds))
    ranks = {}
    for tid in sorted(tasks, reverse=True):
        own = tasks[tid].work / avg_speed
        ranks[tid] = own + max((ranks[c] for c in tasks[tid].children), default=0.0)
    return ranks

print("Workflow generators loaded.")

## Failure model
Ordinary outages follow an alternating renewal model: exponentially distributed healthy uptime, then a fixed repair duration. A correlated shock is added as an exogenous outage interval for selected VMs. Intervals are merged per VM to create a single availability timeline. The generator is deterministic for a given seed. A shock interval's existence is not enough: the simulator tests also verify actual interruption of a running task.

## Optional external-workflow import

The built-in chain/fork-join/random DAGs are **synthetic stress tests**, not a substitute for benchmark evaluation. The helper below accepts a prepared task table and edge list so a public workflow benchmark can be transformed once into a transparent, versioned input. It does not claim to parse every native WfCommons schema automatically. Keep a manifest recording the original dataset URL/version, preprocessing, task-runtime units, and any exclusions.


In [ ]:
# Cell 2b — Optional generic CSV DAG loader for externally sourced workflows
def load_workflow_csv(task_csv_path, edge_csv_path, id_col="task_id",
                      work_col="work", source_col="source", target_col="target"):
    task_df = pd.read_csv(task_csv_path)
    edge_df = pd.read_csv(edge_csv_path)
    required_task = {id_col, work_col}
    required_edge = {source_col, target_col}
    if not required_task.issubset(task_df.columns):
        raise ValueError(f"Task CSV must contain columns {sorted(required_task)}")
    if not required_edge.issubset(edge_df.columns):
        raise ValueError(f"Edge CSV must contain columns {sorted(required_edge)}")
    ids = task_df[id_col].tolist()
    if len(ids) != len(set(ids)):
        raise ValueError("Task identifiers must be unique")
    id_map = {old: i for i, old in enumerate(ids)}
    parents = {i: [] for i in range(len(ids))}
    children = {i: [] for i in range(len(ids))}
    for _, edge in edge_df.iterrows():
        a, b = edge[source_col], edge[target_col]
        if a not in id_map or b not in id_map:
            raise ValueError(f"Edge references unknown task: {(a, b)}")
        ai, bi = id_map[a], id_map[b]
        if ai == bi:
            raise ValueError("Self-loops are invalid")
        children[ai].append(bi)
        parents[bi].append(ai)
    # Topologically reorder nodes so parent IDs precede child IDs.
    import networkx as nx
    graph = nx.DiGraph()
    graph.add_nodes_from(range(len(ids)))
    for a in range(len(ids)):
        graph.add_edges_from((a, b) for b in children[a])
    if not nx.is_directed_acyclic_graph(graph):
        raise ValueError("Imported workflow must be a DAG")
    order = list(nx.topological_sort(graph))
    remap = {old_i: new_i for new_i, old_i in enumerate(order)}
    work_by_old = {id_map[row[id_col]]: float(row[work_col]) for _, row in task_df.iterrows()}
    tasks = {}
    for old_i in order:
        new_i = remap[old_i]
        p = tuple(sorted(remap[x] for x in parents[old_i]))
        c = tuple(sorted(remap[x] for x in children[old_i]))
        tasks[new_i] = Task(new_i, work_by_old[old_i], p, c)
    validate_workflow(tasks)
    return tasks

print("Optional external workflow loader ready.")


In [ ]:
# Cell 3 — Failure-rate profiles and outage trace generation

def nominal_rate(mode: str, vm_id: int, base_rate: float = BASE_FAILURE_RATE) -> float:
    if mode == "none" or base_rate == 0:
        return 0.0
    if mode == "stable":
        return float(base_rate)
    if mode == "hotspot":
        return float(base_rate * (4.0 if vm_id in (0, 1) else 0.55))
    if mode == "shock":
        return float(base_rate * 0.75)
    raise ValueError(f"Unknown failure mode: {mode}")


def merge_intervals(intervals: List[Tuple[float, float]]) -> List[Tuple[float, float]]:
    valid = sorted((float(a), float(b)) for a, b in intervals if b > a)
    if not valid:
        return []
    out = [[valid[0][0], valid[0][1]]]
    for start, end in valid[1:]:
        if start <= out[-1][1] + 1e-12:
            out[-1][1] = max(out[-1][1], end)
        else:
            out.append([start, end])
    return [(a, b) for a, b in out]


def generate_outages(
    mode: str,
    seed: int,
    nominal_makespan: float,
    base_rate: float = BASE_FAILURE_RATE,
    n_vms: int = len(VM_SPEEDS),
) -> Dict[int, List[Tuple[float, float]]]:
    if mode not in ("none", *FAILURE_MODES):
        raise ValueError(f"Unknown mode: {mode}")
    if n_vms <= 0 or nominal_makespan <= 0 or base_rate < 0:
        raise ValueError("n_vms and nominal_makespan must be positive; base_rate nonnegative")
    rng = np.random.default_rng(seed)
    horizon = max(10.0 * float(nominal_makespan), float(nominal_makespan) + 2.0 * REPAIR_TIME)
    traces = {i: [] for i in range(n_vms)}
    for vm_id in range(n_vms):
        rate = nominal_rate(mode, vm_id, base_rate)
        if rate <= 0:
            continue
        # Alternating renewal: each ordinary outage begins after healthy uptime.
        t = float(rng.exponential(1.0 / rate))
        while t < horizon:
            end = t + REPAIR_TIME
            traces[vm_id].append((t, end))
            t = end + float(rng.exponential(1.0 / rate))
    if mode == "shock":
        shock_start = SHOCK_START_FRACTION * float(nominal_makespan)
        shock_duration = max(REPAIR_TIME, SHOCK_DURATION_FRACTION * float(nominal_makespan))
        shock_end = shock_start + shock_duration
        for vm_id in SHOCK_VMS:
            if 0 <= vm_id < n_vms:
                traces[vm_id].append((shock_start, shock_end))
    return {i: merge_intervals(v) for i, v in traces.items()}

print("Failure model loaded.")

## Event-driven execution simulator
The simulator processes task completion and outage start/end events chronologically. If an outage interrupts a running task, completed work since that attempt began is recorded as lost, the remaining work is retained, and the task becomes eligible for rescheduling after the event group is processed. Stale completion events are invalidated with per-task generation tokens. VM outage events are assumed observable through health monitoring, including when the VM is idle; this assumption must be stated in any paper.

In [ ]:
# Cell 4 — Event-driven simulator and scheduling policies

@dataclass
class RuntimeTask:
    remaining: float
    status: str = "pending"  # pending, running, done
    vm_id: Optional[int] = None
    start_time: Optional[float] = None
    token: int = 0
    completion_time: Optional[float] = None
    retry_after: float = 0.0
    interrupted_at: Optional[float] = None

@dataclass
class VMState:
    down: bool = False
    busy_task: Optional[int] = None
    queue: List[int] = field(default_factory=list)
    last_outage_end: float = 0.0
    failure_count: int = 0
    healthy_exposure: float = 0.0
    busy_time: float = 0.0


def estimated_queue_wait(vm_id, now, tasks, runtime, vm_states, speeds):
    """Projected wait from the running task plus all already queued work."""
    vm = vm_states[vm_id]
    wait = 0.0
    if vm.busy_task is not None:
        current = runtime[vm.busy_task]
        elapsed = max(0.0, now - (current.start_time if current.start_time is not None else now))
        remaining = max(0.0, current.remaining - elapsed * speeds[vm_id])
        wait += remaining / speeds[vm_id]
    wait += sum(runtime[tid].remaining / speeds[vm_id] for tid in vm.queue)
    return wait


def choose_vm(scheduler, task_id, now, tasks, runtime, vm_states, speeds, ranks,
              static_rates, prior_rate, deadline, deadline_weight, risk_weight, avg_speed):
    # Busy VMs remain candidates: their queue delay is included in EFT.
    candidates = [i for i, vm in enumerate(vm_states) if not vm.down]
    if not candidates:
        return None
    task = tasks[task_id]
    tail = max(0.0, ranks[task_id] - task.work / avg_speed)
    deadline_aware = scheduler in ("AR-HEFT", "AR-HEFT-NoRisk")
    use_risk = scheduler in ("STATIC-RA-Oracle", "AR-HEFT", "AR-HEFT-NoDeadline")
    duration = runtime[task_id].remaining / 1.0
    waits = {j: estimated_queue_wait(j, now, tasks, runtime, vm_states, speeds) for j in candidates}
    projected = {j: now + waits[j] + runtime[task_id].remaining / speeds[j] + tail for j in candidates}
    feasible = [j for j in candidates if projected[j] <= deadline] if deadline_aware else candidates
    eligible = feasible if feasible else candidates
    scored = []

    for vm_id in eligible:
        duration = runtime[task_id].remaining / speeds[vm_id]
        start_est = now + waits[vm_id]
        finish = start_est + duration
        score = finish
        if use_risk:
            if scheduler in ("AR-HEFT", "AR-HEFT-NoDeadline"):
                alpha0 = max(prior_rate, 0.0) * PRIOR_EXPOSURE
                beta0 = PRIOR_EXPOSURE
                alpha = alpha0 + vm_states[vm_id].failure_count
                beta = beta0 + vm_states[vm_id].healthy_exposure
                hazard = alpha / beta if beta > 0 else prior_rate
            else:
                hazard = static_rates[vm_id]
            p_fail = -math.expm1(-max(0.0, hazard) * duration)
            expected_penalty = p_fail * (0.5 * duration + RESTART_OVERHEAD)
            effective_risk_weight = risk_weight
            if scheduler in ("AR-HEFT", "AR-HEFT-NoDeadline"):
                total_observed = sum(v.failure_count for v in vm_states)
                total_exposure = sum(v.healthy_exposure for v in vm_states)
                observed_rate = total_observed / max(total_exposure, 1e-9)
                if prior_rate > 0:
                    multiplier = 1.0 + ADAPTIVE_WEIGHT_KAPPA * max(0.0, observed_rate / prior_rate - 1.0)
                    effective_risk_weight *= min(MAX_ADAPTIVE_WEIGHT_MULTIPLIER, multiplier)
                # When the best estimated critical-path completion has little
                # deadline slack, reduce the risk tilt so it cannot dominate QoS.
                if deadline_aware and math.isfinite(deadline):
                    best_projected = min(projected.values())
                    task_slack = max(0.0, deadline - best_projected)
                    slack_scale = min(1.0, task_slack / max(DEADLINE_SLACK_GUARD_FRACTION * deadline, 1e-9))
                    effective_risk_weight *= max(0.10, slack_scale)
            score += effective_risk_weight * expected_penalty
        if deadline_aware and not feasible:
            score += deadline_weight * max(0.0, projected[vm_id] - deadline)
        scored.append((score, vm_id))
    return min(scored, key=lambda x: (x[0], x[1]))[1]


def simulate_workflow(tasks, speeds, outages, scheduler="AR-HEFT", deadline=float("inf"),
                      base_rate=BASE_FAILURE_RATE, risk_weight=RISK_WEIGHT,
                      deadline_weight=DEADLINE_WEIGHT, failure_mode="stable"):
    if scheduler not in SCHEDULERS:
        raise ValueError(f"Unknown scheduler: {scheduler}")
    validate_workflow(tasks)
    n_vms = len(speeds)
    runtime = {tid: RuntimeTask(task.work) for tid, task in tasks.items()}
    vm_states = [VMState() for _ in speeds]
    ranks = upward_ranks(tasks, speeds)
    static_rates = [nominal_rate(failure_mode, i, base_rate) for i in range(n_vms)]

    # Event tuple: (time, priority, serial, kind, vm_id, task_id, token).
    # Completion is processed before an outage starting at the same timestamp.
    event_q, serial = [], 0
    def push(t, priority, kind, vm_id=-1, task_id=-1, token=-1):
        nonlocal serial
        serial += 1
        heapq.heappush(event_q, (float(t), priority, serial, kind, vm_id, task_id, token))

    generated_intervals = 0
    for vm_id, intervals in outages.items():
        for start, end in intervals:
            generated_intervals += 1
            push(start, 2, "outage_start", vm_id)
            push(end, 1, "outage_end", vm_id)

    now = 0.0
    useful_work = lost_work = executed_work = 0.0
    restarts = interrupted_failures = observed_outages = 0
    recovery_time = scheduling_runtime = 0.0

    def advance_clock(new_time):
        nonlocal now
        dt = max(0.0, new_time - now)
        if dt:
            for vm in vm_states:
                if not vm.down:
                    vm.healthy_exposure += dt
                if vm.busy_task is not None:
                    vm.busy_time += dt
        now = new_time

    def start_queued():
        # Start the head of each queue whenever a VM is healthy and idle.
        for vm_id, vm in enumerate(vm_states):
            if vm.down or vm.busy_task is not None or not vm.queue:
                continue
            tid = vm.queue.pop(0)
            rt = runtime[tid]
            if rt.interrupted_at is not None:
                nonlocal_recovery = max(0.0, now - rt.interrupted_at)
                # Accumulated below through the outer mutable metric holder.
                metric_holder["recovery_time"] += nonlocal_recovery
                rt.interrupted_at = None
            rt.status = "running"
            rt.vm_id = vm_id
            rt.start_time = now
            rt.token += 1
            vm.busy_task = tid
            push(now + rt.remaining / speeds[vm_id], 0, "task_finish", vm_id, tid, rt.token)

    metric_holder = {"recovery_time": 0.0}

    def dispatch():
        nonlocal scheduling_runtime
        while True:
            ready = [
                tid for tid, rt in runtime.items()
                if rt.status == "pending"
                and rt.retry_after <= now + 1e-10
                and all(runtime[p].status == "done" for p in tasks[tid].parents)
            ]
            if not ready or all(vm.down for vm in vm_states):
                break
            deadline_aware = scheduler in ("AR-HEFT", "AR-HEFT-NoRisk")
            if deadline_aware and math.isfinite(deadline):
                def slack_key(tid):
                    candidate_vms = [j for j, vm in enumerate(vm_states) if not vm.down]
                    if not candidate_vms:
                        return (float("inf"), -ranks[tid], tid)
                    best_finish = min(
                        now + estimated_queue_wait(j, now, tasks, runtime, vm_states, speeds)
                        + runtime[tid].remaining / speeds[j]
                        for j in candidate_vms
                    )
                    tail = max(0.0, ranks[tid] - tasks[tid].work / float(np.mean(speeds)))
                    return (deadline - (best_finish + tail), -ranks[tid], tid)
                tid = min(ready, key=slack_key)
            else:
                tid = sorted(ready, key=lambda x: (-ranks[x], x))[0]
            t0 = time.perf_counter()
            vm_id = choose_vm(scheduler, tid, now, tasks, runtime, vm_states, speeds, ranks,
                              static_rates, base_rate, deadline, deadline_weight, risk_weight, float(np.mean(speeds)))
            scheduling_runtime += time.perf_counter() - t0
            if vm_id is None:
                break
            rt = runtime[tid]
            rt.status = "queued"
            rt.vm_id = vm_id
            vm_states[vm_id].queue.append(tid)
        start_queued()

    start_queued()
    dispatch()
    while any(rt.status != "done" for rt in runtime.values()):
        if not event_q:
            unfinished = [tid for tid, rt in runtime.items() if rt.status != "done"]
            raise RuntimeError(f"Simulation deadlock; unfinished tasks: {unfinished}")
        event_time = event_q[0][0]
        advance_clock(event_time)
        same_time = []
        while event_q and abs(event_q[0][0] - event_time) <= 1e-10:
            same_time.append(heapq.heappop(event_q))
        same_time.sort(key=lambda e: (e[1], e[2]))
        for _, _, _, kind, vm_id, tid, token in same_time:
            if kind == "task_finish":
                rt = runtime[tid]
                if rt.status != "running" or rt.token != token or rt.vm_id != vm_id:
                    continue
                progress = min(rt.remaining, max(0.0, (now - rt.start_time) * speeds[vm_id]))
                executed_work += progress
                useful_work += progress
                rt.remaining = max(0.0, rt.remaining - progress)
                rt.status = "done"
                rt.completion_time = now
                rt.vm_id = None
                rt.start_time = None
                vm_states[vm_id].busy_task = None
            elif kind == "outage_end":
                vm_states[vm_id].down = False
                vm_states[vm_id].last_outage_end = now
            elif kind == "outage_start":
                vm_states[vm_id].down = True
                vm_states[vm_id].failure_count += 1
                observed_outages += 1
                interrupted_tid = vm_states[vm_id].busy_task
                if interrupted_tid is not None:
                    rt = runtime[interrupted_tid]
                    if rt.status == "running":
                        progress = min(rt.remaining, max(0.0, (now - rt.start_time) * speeds[vm_id]))
                        executed_work += progress
                        lost_work += progress
                        # True restart semantics: discard partial progress and
                        # re-execute the full task after the restart delay.
                        rt.remaining = tasks[interrupted_tid].work
                        rt.status = "pending"
                        rt.vm_id = None
                        rt.start_time = None
                        rt.token += 1
                        rt.retry_after = now + RESTART_OVERHEAD
                        rt.interrupted_at = now
                        vm_states[vm_id].busy_task = None
                        restarts += 1
                        interrupted_failures += 1
                        push(rt.retry_after, 3, "retry_ready", -1, interrupted_tid)
            elif kind == "retry_ready":
                pass  # dispatch below will reconsider the pending task

        start_queued()
        dispatch()

    recovery_time = metric_holder["recovery_time"]
    makespan = max(rt.completion_time or 0.0 for rt in runtime.values())
    total_busy = sum(vm.busy_time for vm in vm_states)
    utilization = total_busy / (n_vms * makespan) if makespan > 0 else 0.0
    expected_total_work = sum(task.work for task in tasks.values())
    return {
        "makespan": float(makespan),
        "deadline_violation": int(makespan > deadline),
        "deadline": float(deadline),
        "failures": int(interrupted_failures),
        "restarts": int(restarts),
        "observed_outages": int(observed_outages),
        "generated_outage_intervals": int(generated_intervals),
        "useful_work": float(useful_work),
        "lost_work": float(lost_work),
        "executed_work": float(executed_work),
        "recovery_time": float(recovery_time),
        "utilization": float(utilization),
        "scheduling_runtime_seconds": float(scheduling_runtime),
        "workflow_success": int(all(rt.status == "done" for rt in runtime.values())),
        "useful_work_matches": int(math.isclose(useful_work, expected_total_work, rel_tol=1e-7, abs_tol=1e-7)),
    }

print("Event-driven simulator loaded.")


## Validation tests
These tests check deterministic traces, valid non-overlapping intervals, DAG consistency, exact no-failure execution, and a controlled shock that actually interrupts a running task. A passing test suite validates these specific invariants only; it is not a proof that the model fully represents a real cloud platform.

In [ ]:
# Cell 5 — Simulator tests (must pass before experiments)

def _test_simulator():
    wf = make_workflow("fork_join", 12, seed=77)
    tr1 = generate_outages("shock", 12, 50.0, 0.0, len(VM_SPEEDS))
    tr2 = generate_outages("shock", 12, 50.0, 0.0, len(VM_SPEEDS))
    assert tr1 == tr2, "Trace generation must be deterministic for a fixed seed."
    assert set(tr1) == set(range(len(VM_SPEEDS)))
    for vm_id, intervals in tr1.items():
        for (a, b), (c, d) in zip(intervals, intervals[1:]):
            assert b <= c + 1e-10, "Intervals must be merged and ordered."

    no_outages = {i: [] for i in range(len(VM_SPEEDS))}
    result = simulate_workflow(wf, VM_SPEEDS, no_outages, "HEFT-ECT",
                               deadline=float("inf"), base_rate=0, failure_mode="none")
    assert result["workflow_success"] == 1
    assert result["makespan"] > 0
    assert result["failures"] == 0 and result["lost_work"] == 0
    assert result["useful_work_matches"] == 1
    assert 0 <= result["utilization"] <= 1.0 + 1e-9
    assert math.isclose(result["useful_work"], sum(t.work for t in wf.values()), rel_tol=1e-8)
    no_failure_ar = simulate_workflow(wf, VM_SPEEDS, no_outages, "AR-HEFT",
                                      deadline=float("inf"), base_rate=0.0, failure_mode="none")
    assert math.isclose(no_failure_ar["makespan"], result["makespan"], rel_tol=1e-9), \
        "With zero failure prior and no outages, AR-HEFT should reduce to the time-based baseline."

    # Controlled interruption: the partial work must be discarded and recomputed.
    tiny = {0: Task(0, 100.0, (), ())}
    controlled = {i: [] for i in range(len(VM_SPEEDS))}
    controlled[0] = [(2.0, 8.0)]
    forced_speeds = [10.0] * len(VM_SPEEDS)
    forced = simulate_workflow(tiny, forced_speeds, controlled,
                               "HEFT-ECT", deadline=float("inf"), base_rate=0)
    assert forced["failures"] >= 1 and forced["restarts"] >= 1
    assert forced["lost_work"] > 0
    assert forced["executed_work"] > forced["useful_work"]
    assert forced["useful_work_matches"] == 1
    assert forced["workflow_success"] == 1
    assert forced["recovery_time"] >= RESTART_OVERHEAD

    # Shock intervals are inserted on the intended VMs at the intended time.
    shock = generate_outages("shock", 9, 100.0, 0.0, len(VM_SPEEDS))
    for vm_id in SHOCK_VMS:
        if vm_id < len(VM_SPEEDS):
            assert any(a <= 40.0 < b for a, b in shock[vm_id])

    # Generated shock must interrupt a running task and account for lost work.
    shock_speeds = [10.0] * len(VM_SPEEDS)
    long_task = {0: Task(0, 1000.0, (), ())}
    generated_shock = generate_outages("shock", 9, 100.0, 0.0, len(VM_SPEEDS))
    shock_run = simulate_workflow(long_task, shock_speeds, generated_shock,
                                  "HEFT-ECT", deadline=float("inf"), base_rate=0.0,
                                  failure_mode="shock")
    assert shock_run["failures"] >= 1
    assert shock_run["lost_work"] > 0
    assert shock_run["useful_work_matches"] == 1
    return True

assert _test_simulator()
print("All corrected simulator tests passed.")


## Single-scenario runner
For each workflow configuration, compute an ideal no-outage makespan first. The deadline is then derived from that makespan. All schedulers share the same generated workload and outage trace for each paired seed.

In [ ]:
# Cell 6 — Scenario runner

def run_scenario(structure, n_tasks, seed, mode, deadline_factor=DEADLINE_FACTOR_MAIN,
                 base_rate=BASE_FAILURE_RATE, schedulers=SCHEDULERS):
    workload_seed = int(seed + STRUCTURE_CODE[structure] * 1000 + n_tasks * 17)
    tasks = make_workflow(structure, n_tasks, workload_seed)
    no_outages = {i: [] for i in range(len(VM_SPEEDS))}
    ideal = simulate_workflow(tasks, VM_SPEEDS, no_outages, "HEFT-ECT",
                              deadline=float("inf"), base_rate=0.0, failure_mode="none")
    nominal_makespan = ideal["makespan"]
    deadline = nominal_makespan * deadline_factor
    trace_seed = int(seed + STRUCTURE_CODE[structure] + n_tasks * 13 + MASTER_SEED)
    outages = generate_outages(mode, trace_seed, nominal_makespan, base_rate, len(VM_SPEEDS))
    rows = []
    trace_signature = tuple((vm, tuple(outages[vm])) for vm in sorted(outages))
    for scheduler in schedulers:
        result = simulate_workflow(tasks, VM_SPEEDS, outages, scheduler,
                                   deadline=deadline, base_rate=base_rate, failure_mode=mode)
        rows.append({
            "seed": int(seed), "structure": structure, "n_tasks": int(n_tasks),
            "failure_mode": mode, "base_failure_rate": float(base_rate),
            "deadline_factor": float(deadline_factor), "deadline": float(deadline),
            "scheduler": scheduler, "nominal_makespan": nominal_makespan,
            "trace_seed": trace_seed, "outage_trace_id": f"{structure}:{n_tasks}:{mode}:{trace_seed}",
            "trace_interval_count": sum(len(v) for v in outages.values()),
            **result,
        })
    return rows

smoke_rows = run_scenario("fork_join", 12, seed=1, mode="shock", deadline_factor=1.0)
smoke_df = pd.DataFrame(smoke_rows)
display(smoke_df[["scheduler", "makespan", "deadline_violation", "failures", "restarts",
                  "observed_outages", "lost_work", "recovery_time", "utilization"]].round(4))
# The same workload, deadline, and trace must be shared across all scheduler rows.
assert smoke_df["outage_trace_id"].nunique() == 1
assert smoke_df["deadline"].nunique() == 1
assert len(smoke_rows) == len(SCHEDULERS)
assert smoke_df["workflow_success"].all()
print("Smoke test completed.")

## Main experiment
The default is deliberately disabled. Start with 10 seeds after the tests pass. Each row represents one scheduler run. The same workload seed and outage trace are reused across scheduler variants for each scenario. Avoid changing weights after looking at test-set results; if tuning is required, separate tuning seeds from final evaluation seeds.

In [ ]:
# Cell 7 — Main experiment runner

def run_experiment(n_seeds=N_SEEDS_MAIN, deadline_factor=DEADLINE_FACTOR_MAIN,
                   base_rate=BASE_FAILURE_RATE, structures=WORKFLOW_STRUCTURES,
                   sizes=WORKLOAD_SIZES, modes=FAILURE_MODES, schedulers=SCHEDULERS):
    all_rows = []
    total = len(structures) * len(sizes) * len(modes) * n_seeds
    done = 0
    for structure in structures:
        for n_tasks in sizes:
            for mode in modes:
                for seed in range(n_seeds):
                    all_rows.extend(run_scenario(structure, n_tasks, seed, mode,
                                                 deadline_factor, base_rate, schedulers))
                    done += 1
                print(f"Completed {done}/{total} scenario-seed groups", end="\r")
    print()
    return pd.DataFrame(all_rows)

if RUN_MAIN_EXPERIMENT:
    results = run_experiment(n_seeds=N_SEEDS_MAIN)
    results.to_csv("ar_heft_raw.csv", index=False)
    summary = (results.groupby(["structure", "n_tasks", "failure_mode", "scheduler"])
               .agg(n=("seed", "count"), makespan_mean=("makespan", "mean"),
                    makespan_sd=("makespan", "std"),
                    deadline_violation_rate=("deadline_violation", "mean"),
                    workflow_success_rate=("workflow_success", "mean"),
                    restarts_mean=("restarts", "mean"), failures_mean=("failures", "mean"),
                    lost_work_mean=("lost_work", "mean"), recovery_time_mean=("recovery_time", "mean"),
                    utilization_mean=("utilization", "mean"),
                    scheduling_runtime_mean=("scheduling_runtime_seconds", "mean"))
               .reset_index())
    summary.to_csv("ar_heft_summary.csv", index=False)
    display(summary.head(20).round(4))
    print("Saved ar_heft_raw.csv and ar_heft_summary.csv")
else:
    print("Main experiment disabled. Set RUN_MAIN_EXPERIMENT=True after reviewing the tests and smoke test.")

## Paired statistical analysis
The primary comparison should be specified before final experiments. This cell reports paired mean differences, bootstrap confidence intervals, Wilcoxon signed-rank p-values, and Holm-adjusted p-values for AR-HEFT versus STATIC-RA and HEFT-ECT. Analyses are stratified by workflow structure, size, and failure mode to avoid hiding heterogeneous effects in a pooled mean.

In [ ]:
# Cell 8 — Paired comparisons and multiplicity correction

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    out = np.full(len(p), np.nan)
    valid_idx = np.where(np.isfinite(p))[0]
    if not len(valid_idx):
        return out
    order = valid_idx[np.argsort(p[valid_idx])]
    m = len(order)
    running = 0.0
    for rank, idx in enumerate(order):
        adjusted = min(1.0, (m - rank) * p[idx])
        running = max(running, adjusted)
        out[idx] = running
    return out


def paired_compare(df, metric, comparator, n_boot=4000, seed=2026):
    keys = ["structure", "n_tasks", "failure_mode"]
    rows = []
    rng = np.random.default_rng(seed)
    for group_key, sub in df.groupby(keys):
        wide = sub.pivot_table(index="seed", columns="scheduler", values=metric, aggfunc="first")
        if "AR-HEFT" not in wide or comparator not in wide:
            continue
        wide = wide[["AR-HEFT", comparator]].dropna()
        a = wide["AR-HEFT"].to_numpy(float)
        b = wide[comparator].to_numpy(float)
        d = a - b
        n = len(d)
        if n == 0:
            continue
        if np.allclose(d, 0):
            pval = 1.0
        else:
            try:
                pval = float(stats.wilcoxon(d, zero_method="wilcox", alternative="two-sided").pvalue)
            except ValueError:
                pval = 1.0
        boot = np.mean(rng.choice(d, size=(n_boot, n), replace=True), axis=1)
        lo, hi = np.quantile(boot, [0.025, 0.975])
        rows.append({"structure": group_key[0], "n_tasks": group_key[1], "failure_mode": group_key[2],
                     "metric": metric, "comparator": comparator, "n_pairs": n,
                     "AR_HEFT_mean": float(np.mean(a)), "comparator_mean": float(np.mean(b)),
                     "mean_delta_AR_minus_comparator": float(np.mean(d)),
                     "bootstrap_ci95_low": float(lo), "bootstrap_ci95_high": float(hi),
                     "wilcoxon_p": pval})
    out = pd.DataFrame(rows)
    return out

if "results" in globals():
    comparisons = []
    for metric in ("makespan", "deadline_violation", "restarts", "lost_work", "utilization"):
        for comparator in ("STATIC-RA-Oracle", "HEFT-ECT", "AR-HEFT-NoRisk", "AR-HEFT-NoDeadline"):
            comparisons.append(paired_compare(results, metric, comparator))
    paired_results = pd.concat(comparisons, ignore_index=True)
    # Correct across the full family of reported metric × comparator × workload tests.
    paired_results["holm_p_global"] = holm_adjust(paired_results["wilcoxon_p"].to_numpy())
    paired_results.to_csv("ar_heft_paired_comparisons.csv", index=False)
    display(paired_results.round(5))
else:
    print("Run the main experiment before paired analysis.")

## Visualization
Plots use the saved raw run-level results. Treat them as descriptive; statistical inference comes from the paired analysis and should be interpreted alongside effect sizes and uncertainty.

In [ ]:
# Cell 9 — Main plots
if "results" in globals():
    plot_metrics = ["makespan", "deadline_violation", "restarts", "lost_work", "recovery_time", "utilization", "scheduling_runtime_seconds"]
    for metric in plot_metrics:
        grouped = (results.groupby(["failure_mode", "scheduler"])[metric]
                   .agg(["mean", "std"]).reset_index())
        fig, ax = plt.subplots(figsize=(9, 4.5))
        for scheduler in SCHEDULERS:
            s = grouped[grouped["scheduler"] == scheduler]
            ax.plot(s["failure_mode"], s["mean"], marker="o", label=scheduler)
        ax.set_title(f"Mean {metric.replace('_', ' ')} by failure mode")
        ax.set_xlabel("Failure mode")
        ax.set_ylabel(metric.replace("_", " "))
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2, fontsize=8)
        plt.tight_layout()
        plt.savefig(f"ar_heft_{metric}.png", dpi=180)
        plt.show()
else:
    print("Run the main experiment before plotting.")

## Parameter sensitivity
This experiment reruns scenarios with different deadline factors and base failure intensities. It is disabled by default because it adds substantial computation. Calibrate factors using pilot seeds, then freeze the chosen range and report all tested factors. Do not select only the factor that favors AR-HEFT.

In [ ]:
# Cell 10 — Optional deadline sensitivity experiment

def run_deadline_sensitivity(n_seeds=N_SEEDS_SENSITIVITY, factors=DEADLINE_FACTORS):
    frames = []
    for factor in factors:
        print(f"Deadline factor {factor}")
        frame = run_experiment(n_seeds=n_seeds, deadline_factor=float(factor))
        frames.append(frame)
    raw = pd.concat(frames, ignore_index=True)
    raw.to_csv("ar_heft_deadline_sensitivity_raw.csv", index=False)
    summary = (raw.groupby(["deadline_factor", "failure_mode", "scheduler"])
               .agg(n=("seed", "count"), makespan_mean=("makespan", "mean"),
                    deadline_violation_rate=("deadline_violation", "mean"),
                    restarts_mean=("restarts", "mean"), lost_work_mean=("lost_work", "mean"))
               .reset_index())
    summary.to_csv("ar_heft_deadline_sensitivity_summary.csv", index=False)
    return raw, summary

if RUN_DEADLINE_SENSITIVITY:
    sensitivity_raw, sensitivity_summary = run_deadline_sensitivity()
    display(sensitivity_summary.round(4))
else:
    print("Deadline sensitivity disabled. Enable after the main pilot is validated.")

## Research interpretation and required reporting

Before a submission, document at minimum:
- Why the outage process, repair duration, shock scope, and monitoring assumptions are appropriate.
- Whether failure rates are calibrated to real traces or are synthetic stress-test parameters.
- How the online hazard estimator updates and when the scheduler is allowed to observe outages.
- Exact definitions of useful work, lost work, utilization, workflow success, and deadline success.
- Baseline implementations and deviations from canonical HEFT or published methods.
- A pre-specified primary metric, paired experimental design, confidence intervals, effect sizes, and multiplicity correction.
- Ablations, workload-size and topology sensitivity, parameter sensitivity, runtime overhead, and failure cases.
- Reproducibility: seeds, versions, configuration, raw CSVs, and code revision.

### Known limitations of this framework
- Synthetic DAGs only; no established benchmark suite or real failure traces are included by default.
- Communication costs, checkpointing, migration overhead, billing, and network contention are not modeled.
- The shock is a fixed exogenous interval affecting a fixed VM subset.
- The adaptive hazard estimator uses a Gamma-Poisson-style posterior mean with healthy-time exposure; it assumes observed outage starts are available through monitoring.
- HEFT-ECT is a computation-only HEFT-style baseline, not canonical HEFT with communication costs.
- A simulation study alone does not establish novelty. Review recent related work and compare against the closest validated baselines before making a publication claim.

**Do not claim that this notebook is publication-ready merely because tests pass.** It is a more defensible experimental foundation whose assumptions and results still require independent validation.

## Publication gate: do not skip

The current simulator is intentionally more correct than the prior draft, but the notebook still does **not** by itself justify a journal submission. Before making claims:

1. Run at least 100 paired seeds per configuration and save raw run-level CSVs.
2. Use a real benchmark workflow set (e.g., WfCommons/Pegasus-derived DAGs) in addition to synthetic DAGs; publish the benchmark manifest and preprocessing script.
3. Add communication/data-transfer costs if the selected benchmark and claim require them.
4. Compare with faithful implementations or clearly labeled adapted baselines. In particular, this notebook does not implement TFR-GNN, uHEFT, CRFTS, or RWSUT.
5. Report the full distribution and paired effect sizes, confidence intervals, and global multiplicity-corrected p-values—not only averages.
6. Report sensitivity to deadline factor, base failure intensity, prior exposure, risk weight, restart overhead, and shock timing.
7. Keep the primary endpoint and statistical test fixed before looking at the final runs.
8. Include a no-failure sanity test: the reliability-aware method should not claim a reliability benefit when no failures occur.
9. Describe assumptions explicitly: exponential healthy uptime, fixed repair time, synthetic correlated shock, no migration/checkpoint state, no communication cost, and single-workflow evaluation.
10. If AR-HEFT does not outperform a baseline, report that result and narrow the contribution to a supported trade-off or methodological finding.

### Corrected interpretation of failure metrics

- `failures` / `restarts`: number of running-task interruptions that caused a restart.
- `observed_outages`: all outage-start events processed before workflow completion, including outages on idle VMs.
- `lost_work`: computation discarded when interrupted tasks restart from their full work requirement.
- `executed_work`: useful plus discarded computation.
- `recovery_time`: elapsed simulation time from interruption to the next start of that task.
- `utilization`: aggregate busy time divided by VM count × workflow makespan; it includes useful execution and partial work before interruptions.
- `deadline_violation`: binary indicator for workflow makespan greater than the common deadline.
